# Model Training Pipeline
This notebook trains an XGBoost regression model to predict `target_water_level_next` using the engineered dataset.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


### 1. Load Preprocessed Data
Load `processed_training_data.csv` which was generated in the preprocessing pipeline.


In [ ]:
# It assumes you ran preprocessing.ipynb first to generate processed_training_data.csv
df = pd.read_csv('processed_training_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
print(f"Data shape: {df.shape}")
df.head()


### 2. Feature Selection
Define the strict list of features to use. Drop strings, raw datetime, and the target variable itself.


In [ ]:
target_col = 'target_water_level_next'

# We want to drop string columns and the target
string_cols = ['station', 'river_basin', 'rainfall_type', 'status', 'datetime']
drop_cols = string_cols + [target_col]

features = [col for col in df.columns if col not in drop_cols]
print(f"Number of input features: {len(features)}")
print("Features:", features)


### 3. Time-Based Train-Test Split
In time-series, we NEVER random split. We'll train on everything before Jan 1st 2025, and use data after as the test set.


In [ ]:
train_mask = df['datetime'] < '2025-01-01'
test_mask = df['datetime'] >= '2025-01-01'

df_train = df[train_mask]
df_test = df[test_mask]

X_train = df_train[features]
y_train = df_train[target_col]

X_test = df_test[features]
y_test = df_test[target_col]

print(f"Training sequences: {len(X_train)}")
print(f"Testing sequences: {len(X_test)}")


### 4. Initialize & Train Model
We use XGBoost Regressor because it cleanly handles missing inputs and seamlessly models non-linear interactions without scaling.


In [ ]:
# simple, robust initial configuration
model = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model trained successfully!")


### 5. Predict and Evaluate
Compute RMSE (Root Mean Squared Error) and MAE (Mean Absolute Error).


In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")


### 6. Error Analysis
Look at error distribution across stations to identify struggling areas.


In [ ]:
df_test = df_test.copy()
df_test['predicted_water_level'] = y_pred
df_test['error_abs'] = np.abs(df_test['target_water_level_next'] - df_test['predicted_water_level'])

# Average error per station
station_errors = df_test.groupby('station')['error_abs'].mean().sort_values(ascending=False)
print("Mean Absolute Error by Station:")
print(station_errors)


### 7. Feature Importance
Looking at which features drive the model decisions.


In [ ]:
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'][:15][::-1], feature_importance['importance'][:15][::-1])
plt.title('Top 15 Most Important Features')
plt.xlabel('XGBoost Feature Importance')
plt.show()


### 8. Save Model
Export the trained model so the FastAPI backend can load it for real-time inference.


In [ ]:
os.makedirs('models', exist_ok=True)
model_path = 'models/waterlevel_xgb_v1.joblib'
joblib.dump(model, model_path)

print(f"Model saved efficiently to {model_path}")
print("Done! Proceed to the next phase in your project.")
